Homework: Agentic RAG

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

GithubRepositoryDataReader downloads the entire repository and goes over all the files in it. Because we specify allowed_extensions={"md"}, it only checks the markdown files.

We also pass a filename_filter so we don't grab every markdown file in the repo, like the top-level README. The lesson pages all live under a module's lessons/ folder, so filtering on /lessons/ keeps just those.

Each file has a parse() method that returns a dictionary with its filename and content:

Q1. How many lesson pages

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

Q2. Indexing and searching

In [4]:
import minsearch

index = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)
results = index.search("How does the agentic loop keep calling the model until it stops?", num_results=1)

print(results[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


In [7]:
import os
from google import genai
from dotenv import load_dotenv
load_dotenv()

# 1. Initialize the native Gemini Client
# It automatically looks for the GEMINI_API_KEY environment variable
gemini_client = genai.Client()

# 2. Define the Homework RAG Class
class HomeworkRAG:
    def __init__(self, index, llm_client):
        self.index = index
        self.llm_client = llm_client
        
    def search(self, query):
        return self.index.search(query, num_results=5)
        
    def build_prompt(self, query, search_results):
        context = ""
        for doc in search_results:
            context += f"filename: {doc['filename']}\ncontent: {doc['content']}\n\n---\n\n"
            
        prompt = f"""
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT.

QUESTION: {query}

CONTEXT:
{context}
""".strip()
        return prompt
        
    def llm(self, prompt):
        # Using gemini-2.5-flash as an efficient, token-accurate model for this task
        response = self.llm_client.models.generate_content(
            model="gemini-2.5-flash", 
            contents=prompt
        )
        return response
        
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        
        # Extract the input (prompt) tokens from Gemini's usage metadata
        input_tokens = response.usage_metadata.prompt_token_count
        
        return response.text, input_tokens

# --- Execution ---

# Initialize the custom RAG with your Q2 index (ensure your 'index' variable from Q2 is ready)
hw_rag = HomeworkRAG(index=index, llm_client=gemini_client)

# Run the specific homework query
query = "How does the agentic loop keep calling the model until it stops?"
answer, tokens = hw_rag.rag(query)

print(f"Input (Prompt) Tokens: {tokens}")
print("\nAssistant Answer:\n", answer)

Input (Prompt) Tokens: 7933

Assistant Answer:
 The agentic loop keeps calling the model until it stops by using a `while` loop that continues as long as the model indicates it needs to perform more actions (specifically, make more function calls).

Here's how it works:
1.  **Initial Call**: The loop starts by sending the current message history (including instructions and the user's question) to the LLM.
2.  **Process Response**: The LLM's response is processed.
    *   If the LLM's response includes a `function_call` (e.g., to use the `search` tool), the code executes that function call. The output of the function call is then appended to the message history. A flag, `has_function_calls`, is set to `True`.
    *   If the LLM's response contains only a `message` (i.e., a final answer) and no function calls, the `has_function_calls` flag remains `False`.
3.  **Loop Condition**: After processing the response, the loop checks the `has_function_calls` flag.
    *   If `has_function_calls`

Q4. Chunking

In [8]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [9]:
len(chunks)

295

Q5. RAG with chunking

In [10]:
# 1. Initialize a new index specifically for chunks
chunk_index = minsearch.Index(
    text_fields=["content"], 
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

# 2. Point your custom HomeworkRAG class at the new chunked index
chunk_rag = HomeworkRAG(index=chunk_index, llm_client=gemini_client)

# 3. Run the exact same query over the chunked context
query_q5 = "How does the agentic loop keep calling the model until it stops?"
answer_q5, tokens_q5 = chunk_rag.rag(query_q5)

print(f"Chunked Input (Prompt) Tokens: {tokens_q5}")

Chunked Input (Prompt) Tokens: 2586


Q6. Turning it into an agent

In [11]:
# 1. Define the search tool function with appropriate type hints and a docstring
def search_chunked_index(query: str) -> list:
    """
    Search the chunked lesson database for entries matching the given query keywords.
    """
    # Look up matching chunks from the chunk index we made in Q5
    return chunk_index.search(query, num_results=5)

# 2. Set up the agent instructions guiding it to make multiple searches
agent_instructions = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
"""

# 3. Create the agent configuration containing our instructions and search function tool
agent_config = genai.types.GenerateContentConfig(
    system_instruction=agent_instructions,
    tools=[search_chunked_index],
)

# 4. Spin up the native Gemini chat agent session
chat_agent = gemini_client.chats.create(
    model="gemini-2.5-pro",  # Using Pro for highly reliable tool execution
    config=agent_config
)

# 5. Send the target prompt
agent_query = "How does the agentic loop work, and how is it different from plain RAG?"
print("Sending prompt to the agent...")
response_agent = chat_agent.send_message(agent_query)

# 6. Count how many times the agent used the search function tool
history = chat_agent.get_history()
function_call_count = 0

for message in history:
    for part in message.parts:
        # Check if the part represents a tool call requested by the model
        if part.function_call:
            function_call_count += 1
            print(f"Search call {function_call_count}: args -> {part.function_call.args}")

print(f"\nTotal times the agent called the search tool: {function_call_count}")

Sending prompt to the agent...
Search call 1: args -> {'query': 'agentic loop'}
Search call 2: args -> {'query': 'Retrieval-Augmented Generation RAG'}
Search call 3: args -> {'query': 'agentic loop versus RAG'}

Total times the agent called the search tool: 3
